In [1]:
!uv pip install "zenml[server]" pandas --quiet
# !zenml integration install sklearn --uv


In [2]:
!rm -rf .zen
!zenml init

⠋ Initializing ZenML repository at /Users/kevin/Desktop/ai-engineer/mlops/zenml.
⠙ Initializing ZenML repository at /Users/kevin/Desktop/ai-engineer/mlops/zenml.
⠹ Initializing ZenML repository at /Users/kevin/Desktop/ai-engineer/mlops/zenml.
⠸ Initializing ZenML repository at /Users/kevin/Desktop/ai-engineer/mlops/zenml.
⠼ Initializing ZenML repository at /Users/kevin/Desktop/ai-engineer/mlops/zenml.
⠴ Initializing ZenML repository at /Users/kevin/Desktop/ai-engineer/mlops/zenml.
⠦ Initializing ZenML repository at /Users/kevin/Desktop/ai-engineer/mlops/zenml.
Setting the repo active project to 'default'.
Setting the repo active stack to default.
ZenML repository initialized at /Users/kevin/Desktop/ai-engineer/mlops/zenml.
⠧ Initializing ZenML repository at /Users/kevin/Desktop/ai-engineer/mlops/zenml.
⠧ Initializing ZenML repository at /Users/kevin/Desktop/ai-engineer/mlops/zenml.

The local active stack was initialized to 'default'. This local configuration 
will only take effect whe

In [3]:
import numpy as np
from sklearn.base import ClassifierMixin
from sklearn.svm import SVC
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

def train_test() -> None:
    """ Train and test a Scikit-learn SVC classifier on digits """
    digits = load_digits()
    # print(digits)

    data = digits.images.reshape((len(digits.images), -1))
    # print(data)

    X_train, X_test, y_train, y_test = train_test_split(
        data, digits.target, test_size=0.2, shuffle=False
    )
    model = SVC(gamma=0.001)
    model.fit(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    print(f"Test accuracy: {test_acc}")


train_test()

Test accuracy: 0.9583333333333334


In [4]:
from zenml import step
from typing_extensions import Annotated
import pandas as pd
from typing import Tuple

@step
def importer() -> Tuple[
    Annotated[np.ndarray, "X_train"],
    Annotated[np.ndarray, "X_test"],
    Annotated[np.ndarray, "y_train"],
    Annotated[np.ndarray, "y_test"]
]:
    """ Load the digits dataset as numpy arrays """
    digits = load_digits()
    data = digits.images.reshape((len(digits.images), -1))
    X_train, X_test, y_train, y_test = train_test_split(
        data, digits.target, test_size=0.2, shuffle=False
    )
    return X_train, X_test, y_train, y_test

In [5]:
@step
def svc_trainer(
    X_train: np.ndarray,
    y_train: np.ndarray
) -> ClassifierMixin:
    """ Train an sklearn SVC classifier """
    model = SVC(gamma=0.001)
    model.fit(X_train, y_train)
    return model

In [6]:
@step
def evaluator(
    X_test: np.ndarray,
    y_test: np.ndarray,
    model: ClassifierMixin
):
    """ Calculator the test set accuracy of an sklearn model """
    test_acc = model.score(X_test, y_test)
    print(f"Test accuracy: {test_acc}")
    return test_acc

In [7]:
from zenml import pipeline

@pipeline
def digits_pipeline():
    """ Links all the steps together in a pipeline """
    X_train, X_test, y_train, y_test = importer()
    model = svc_trainer(X_train=X_train, y_train=y_train)
    evaluator(X_test=X_test, y_test=y_test, model=model)

In [8]:
digits_svc_pipeline = digits_pipeline()

Initiating a new run for the pipeline: digits_pipeline.


Using Python 3.11.14 environment at: /Users/kevin/Desktop/ai-engineer/mlops/.venv


Registered new pipeline: digits_pipeline.
Using user: default
Using stack: default
  orchestrator: default
  deployer: default
  artifact_store: default
You can visualize your pipeline runs in the ZenML Dashboard. In order to try it locally, please run zenml login --local.
Step importer has started.


Using Python 3.11.14 environment at: /Users/kevin/Desktop/ai-engineer/mlops/.venv


Skipping visualization of numpy array because matplotlib is not installed. To install matplotlib, run pip install matplotlib.
Skipping visualization of numpy array because matplotlib is not installed. To install matplotlib, run pip install matplotlib.
Step importer has finished in 0.802s.
Step svc_trainer has started.
Step svc_trainer has finished in 0.122s.
Step evaluator has started.


Using Python 3.11.14 environment at: /Users/kevin/Desktop/ai-engineer/mlops/.venv
Using Python 3.11.14 environment at: /Users/kevin/Desktop/ai-engineer/mlops/.venv


Test accuracy: 0.9583333333333334
Step evaluator has finished in 0.653s.
Pipeline run has finished in 1.721s.


In [ ]:
# !zenml login --local
# export OBJC_DISABLE_INITIALIZE_FORK_SAFETY=YES && zenml login --local

Error: The `OBJC_DISABLE_INITIALIZE_FORK_SAFETY` environment variable is recommended to run the ZenML server locally on a Mac. Please set it to `YES` and try again.
